# 02 — Explore P-4 Operator/Lease Database

**Phase 1 — Data Discovery and Understanding**

This notebook is exploratory only. It answers:

- What does one row represent?
- How many records exist?
- What fields exist, and what are their data types?
- Are there missing values?
- What date range is covered?
- Are there duplicate records?
- Are there invalid values?

Source file: `data/raw/operators/p4f606.ebc` — an EBCDIC-encoded,
hierarchical mainframe extract of RRC's "Complete P-4" tape (Producer's
Transportation Authority and Certificate of Compliance), downloaded from
https://www.rrc.texas.gov/resource-center/research/data-sets-available-for-download/.
Format is documented in
`data/raw/operators/p4-user-manual_p4a002_feb2015.pdf` and summarized in
`../docs/data_p4_operators.md`; read that first for the full segment layout.

Key facts recap:
- Fixed **92-byte** binary records, EBCDIC encoded (assumed code page 037,
  spot-checked against decoded data — see `../docs/data_p4_operators.md`)
- No delimiters — records must be read in fixed strides
- **30 possible segment types** multiplexed in one file (vs. 24 on the
  production tape), identified by a 2-byte numeric key at the start of
  each record
- This notebook focuses on the **Root segment (01)** only — it carries
  the lease key (`district_code` + `lease_nbr`, matching the production
  tape's key exactly) and the current `operator_number`, which is all
  that's needed to bridge production leases to P-5 organization records
  (company names)
- The raw file is **2.7 GB / ~30.3M records** — much larger than the
  production tape. This notebook streams through the entire file once
  (chunked reads) to get exact record-type counts and Root data, but uses
  a **bounded sample** (not a full second scan) for the one question that
  needs a segment we don't otherwise parse (P4INFO filing dates) — flagged
  clearly where that happens.

In [2]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path("..") / "src"))

from oil_pipeline.extract.p4_operators import RECORD_LENGTH, load_p4f606

RAW_DATA_PATH = Path("..") / "data" / "raw" / "operators" / "p4f606.ebc"
RAW_DATA_PATH.resolve()


WindowsPath('C:/texas-oil-data-platform/data/raw/operators/p4f606.ebc')

## File size & record count

Confirm the file divides evenly into 92-byte records (validates the fixed-length assumption from the format spec) before reading anything.

In [3]:
file_size = RAW_DATA_PATH.stat().st_size
total_records, remainder = divmod(file_size, RECORD_LENGTH)

print(f"File size:        {file_size:,} bytes")
print(f"Record length:    {RECORD_LENGTH} bytes")
print(f"Total records:    {total_records:,}")
print(f"Remainder bytes:  {remainder} (should be 0 for a clean fixed-length file)")


File size:        2,787,886,120 bytes
Record length:    92 bytes
Total records:    30,303,110
Remainder bytes:  0 (should be 0 for a clean fixed-length file)


## What does one row represent?

One physical 92-byte record is **one segment of one of 30 possible types**
(see `../docs/data_p4_operators.md`), not one lease. The file reading and
EBCDIC decoding logic lives in `src/oil_pipeline/` (`extract/p4_operators.py`,
`utils.py`), not in this notebook — `load_p4f606` below streams the entire
file once, tallying every record by its 2-byte type key and parsing the
Root segment as it goes.

## Scan the full file

`load_p4f606` (in `src/oil_pipeline/extract/p4_operators.py`) streams all
~30.3M records in a single pass: tallies every record by type key, and
parses Root segment records into a DataFrame. This is the expensive cell
in this notebook (reads the full 2.7 GB file) — everything after it
just analyzes the results in memory.

In [4]:
import logging

logging.basicConfig(level=logging.INFO, format="%(message)s")

results = load_p4f606(RAW_DATA_PATH)

key_counts = results["key_counts"]
df_root = results["root"]

assert key_counts["count"].sum() == total_records, "scanned count should match the file-size-derived total"


...5,000,000 records scanned
...10,000,000 records scanned
...15,000,000 records scanned
...20,000,000 records scanned
...25,000,000 records scanned
...30,000,000 records scanned
Done: 30,303,110 records scanned, 548,100 Root records found


In [5]:
key_counts


,key,count,segment,pct
0,03,12128749,P4GPN (Gatherer/Purchaser/Nominator),40.02
1,10,4759363,P4SEVRMK (Severance Remarks),15.71
2,02,3766597,P4INFO (General P-4 Filing Information),12.43
3,04,3340780,P4REMARK (P-4 Remarks),11.02
4,16,2531573,P4OSCHCY (Oil Schedule Cycle),8.35
5,09,1043593,P4SEVR (Severance Information),3.44
6,07,579959,P4LSENM (Lease Name),1.91
7,01,548100,P4ROOT (Root Segment),1.81
8,11,294583,P4GSCHED (Gas Schedule Information),0.97
9,15,253514,P4OSCHED (Oil Schedule Information),0.84


In [6]:
print(f"Root (01): {len(df_root):,} rows")


Root (01): 548,100 rows


## What fields exist, and what are their data types?

Preview the parsed Root segment. All columns come out as Python `str`
from the parser; `.info()` shows the resulting pandas dtypes.

In [7]:
df_root.head()


,oil_gas_code,district_code,lease_nbr,operator_number
0,G,01,000001,739425
1,G,01,000002,342800
2,G,01,000003,245800
3,G,01,000004,302450
4,G,01,000005,357100


In [8]:
df_root.info()


<class 'pandas.DataFrame'>
RangeIndex: 548100 entries, 0 to 548099
Data columns (total 4 columns):
 #   Column           Non-Null Count   Dtype
---  ------           --------------   -----
 0   oil_gas_code     548100 non-null  str  
 1   district_code    548100 non-null  str  
 2   lease_nbr        548100 non-null  str  
 3   operator_number  548100 non-null  str  
dtypes: str(4)
memory usage: 24.6 MB


## Are there missing values?

This is a fixed-format binary extract — every field is present in every
record by construction (no free-form nulls). "Missing" here instead means
a zero-filled `operator_number` (`000000` = no operator currently assigned
to the lease).

In [9]:
print("Null counts (should all be 0 — fixed binary layout has no free-form nulls):")
print(df_root.isna().sum())
print()
no_operator = (df_root["operator_number"] == "000000").sum()
print(f"Root records with operator_number == '000000' (no operator assigned): {no_operator:,} / {len(df_root):,}")


Null counts (should all be 0 — fixed binary layout has no free-form nulls):
oil_gas_code       0
district_code      0
lease_nbr          0
operator_number    0
dtype: int64

Root records with operator_number == '000000' (no operator assigned): 0 / 548,100


## What date range is covered?

The Root segment we parse doesn't carry any date fields — filing dates
live on the **P4INFO segment (key `02`)**, which `load_p4f606` doesn't
parse (not needed for the operator-number join). Per
`../docs/data_p4_operators.md`, `P4-EFFECTIVE-YEAR` sits at byte position
19–22 (1-indexed) within that segment.

This is exploratory-only code, not part of the reusable pipeline in
`src/oil_pipeline/`, so it's written directly here using the shared
`iter_records`/`decode_text` primitives. To keep this fast on a 2.7 GB
file, it scans a **bounded sample** (first 5M records) rather than the
full file — labeled clearly as a sample, not an exact count like the
Root/key_counts numbers above.

In [10]:
from itertools import islice

from oil_pipeline.utils import decode_text, iter_records

SAMPLE_SIZE = 5_000_000
P4INFO_KEY = "02"

effective_years = []
for record in islice(iter_records(RAW_DATA_PATH, RECORD_LENGTH), SAMPLE_SIZE):
    if decode_text(record[0:2]) == P4INFO_KEY:
        effective_years.append(decode_text(record[18:22]))

effective_years = pd.Series(effective_years)
print(f"P4INFO records in first {SAMPLE_SIZE:,} records: {len(effective_years):,}")
print()
print("P4-EFFECTIVE-YEAR distribution (sample only):")
print(effective_years.value_counts().sort_index())


P4INFO records in first 5,000,000 records: 596,644

P4-EFFECTIVE-YEAR distribution (sample only):
0000    26103
0025        1
1911        1
1920        1
1932        2
        ...  
2022    10970
2023    20722
2024     5478
2025     9810
2026     4737
Name: count, Length: 70, dtype: int64


## Are there duplicate records?

`P4-ROOT-KEY` = `oil_gas_code` + `district_code` + `lease_nbr` should be
unique across the whole file — one Root record per lease/well.

In [11]:
dup_leases = df_root.duplicated(subset=["oil_gas_code", "district_code", "lease_nbr"]).sum()
print(f"Duplicate (oil_gas_code, district_code, lease_nbr) keys: {dup_leases:,} / {len(df_root):,}")

exact_dup_root_rows = df_root.duplicated().sum()
print(f"Fully duplicate Root rows: {exact_dup_root_rows:,}")


Duplicate (oil_gas_code, district_code, lease_nbr) keys: 1 / 548,100
Fully duplicate Root rows: 1


## Are there invalid values?

Sanity-check values against what the format spec says they should be:
- `oil_gas_code` should be `O` or `G`
- `district_code` should be one of the 14 known encoded district values
  (same encoding as the production tape — see `../docs/data_p4_operators.md`)
- `operator_number` should be 6 numeric digits

In [12]:
# Same 14 stored values as the production tape's district_code (both tapes
# share RRC's district encoding convention); "12" is documented as unused.
VALID_DISTRICT_CODES = {f"{i:02d}" for i in range(1, 15)}

invalid_oil_gas_code = (~df_root["oil_gas_code"].isin(["O", "G"])).sum()
invalid_district = (~df_root["district_code"].isin(VALID_DISTRICT_CODES)).sum()
invalid_operator_number = (~df_root["operator_number"].str.fullmatch(r"\d{6}")).sum()

print(f"Root — oil_gas_code not O/G:           {invalid_oil_gas_code:,} / {len(df_root):,}")
print(f"Root — unrecognized district_code:     {invalid_district:,} / {len(df_root):,}")
print(f"Root — operator_number not 6 digits:   {invalid_operator_number:,} / {len(df_root):,}")


Root — oil_gas_code not O/G:           0 / 548,100
Root — unrecognized district_code:     0 / 548,100
Root — operator_number not 6 digits:   0 / 548,100


## Summary & next steps

Full-file scan of all 30,303,110 records:

- One physical row = one 92-byte record of 1 of 30 types, not one lease —
  far more segment types than the production tape (24), but this notebook
  only needed the Root segment (01) for the operator-number join.
- Root (01): **548,100 records** — 1.81% of the file. This is much bigger
  than the production tape's 165,436 oil leases because P-4 covers **both
  oil leases and gas wells** statewide, not oil only.
- Segment breakdown is dominated by P4GPN (Gatherer/Purchaser/Nominator,
  40.0%), P4SEVRMK (15.7%), P4INFO (12.4%), and P4REMARK (11.0%) — these
  are all per-filing/audit-trail segments, so they vastly outnumber the
  one-per-lease Root segment.
- No null values are possible in this fixed binary layout, and **every**
  Root record has a real (non-zero) `operator_number` — no leases with
  "no operator assigned" in this file.
- **One exact duplicate Root row found** (`oil_gas_code` + `district_code`
  + `lease_nbr` collision) — a real, tiny anomaly (1 out of 548,100), not
  investigated further but worth knowing about if an exact 1:1 lease-to-row
  assumption ever matters downstream.
- `P4-EFFECTIVE-YEAR` (sampled from the first 5M records, not full-file):
  mostly sensible, but with real noise — a large `0000` bucket (documented:
  effective date isn't required for a lease's first P-4 filing), a handful
  of clearly wrong outliers (`0025`, `1911`, `1920`, `1932`), and some
  **future-dated entries** (2024–2026) — legitimate per the spec, since
  operators can file a P-4 with an effective date they're requesting in
  advance.
- Zero invalid `oil_gas_code`, `district_code`, or `operator_number`
  values found.

The file-reading and decoding logic lives in
`src/oil_pipeline/extract/p4_operators.py` (`load_p4f606`) and
`src/oil_pipeline/utils.py` — matching the pattern from
`01_explore_rrc_production.ipynb`. This data already flows into
`lease_operators` via `oil_pipeline/transform.py` and `main.py` — see
`03_explore_p5_organizations.ipynb` for the other half of that join
(operator number → company name).